# 04 - Decision Tree\n
\n
Arvore de decisao em Spark MLlib, respeitando a restricao do enunciado de nao usar ensembles.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/notebooks')

import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.regression import DecisionTreeRegressor

from _lib import (
    build_spark, append_metric,
    SEED, SPLIT_DATE, SILVER_PATH, TARGET_COL,
    NUMERIC_COLS, CATEGORICAL_COLS,
)

MODEL_OUTPUT = '/models/decision_tree'
IMPORTANCE_PLOT = '/results/feature_importance_decision_tree.png'

spark = build_spark('nyc-rideshare-decision-tree')
sns.set_theme(style='whitegrid')

In [ ]:
silver_df = spark.read.parquet(SILVER_PATH)
silver_df.createOrReplaceTempView('trips_silver')

# Split temporal via Spark SQL puro (regra do projeto: data prep em SQL, ML pipeline em Python).
train_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime < TIMESTAMP '{SPLIT_DATE}'
""").cache()
test_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime >= TIMESTAMP '{SPLIT_DATE}'
""").cache()

train_rows = train_df.count()
test_rows = test_df.count()
if train_rows == 0 or test_rows == 0:
    raise ValueError('O split temporal exige dados antes e depois de 2023-06-01. Um unico mes como 2023-08 nao basta para treinar e avaliar.')

train_rows, test_rows

In [ ]:
indexers = [
    StringIndexer(inputCol=col_name, outputCol=f'{col_name}_idx', handleInvalid='keep')
    for col_name in CATEGORICAL_COLS
]
feature_cols = NUMERIC_COLS + [f'{col_name}_idx' for col_name in CATEGORICAL_COLS]
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
# VectorIndexer marca features com <= maxCategories valores unicos como categoricas.
# Cobre license_num (4), pu/do_borough (6 boroughs + Unknown via handleInvalid='keep' = 7),
# alem de marcar as binarias (is_weekend, is_rush_hour, etc.) como categoricas - efeito benigno.
vector_indexer = VectorIndexer(inputCol='features_raw', outputCol='features', maxCategories=8)
dt = DecisionTreeRegressor(
    labelCol=TARGET_COL,
    featuresCol='features',
    predictionCol='prediction',
    maxDepth=8,
    maxBins=256,
    seed=SEED
)

pipeline = Pipeline(stages=indexers + [assembler, vector_indexer, dt])

In [ ]:
start_time = time.perf_counter()\n
dt_model = pipeline.fit(train_df)\n
train_seconds = time.perf_counter() - start_time\n
\n
dt_model.write().overwrite().save(MODEL_OUTPUT)\n
print(f'Modelo salvo em {MODEL_OUTPUT} | treino: {train_seconds:.2f}s')

In [ ]:
predictions = dt_model.transform(test_df).cache()\n
evaluators = {\n
    'rmse': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='rmse'),\n
    'mae': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='mae'),\n
    'r2': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='r2')\n
}\n
metrics = {name: evaluator.evaluate(predictions) for name, evaluator in evaluators.items()}\n
metrics

In [ ]:
predictions.createOrReplaceTempView('predictions_dt')\n
\n
spark.sql("""\n
SELECT\n
    pu_borough,\n
    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n
    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n
    COUNT(*) AS rows\n
FROM predictions_dt\n
GROUP BY 1\n
ORDER BY rmse DESC\n
""").show(truncate=False)\n
\n
spark.sql("""\n
SELECT\n
    CASE\n
        WHEN trip_miles < 2 THEN '0-2 mi'\n
        WHEN trip_miles < 5 THEN '2-5 mi'\n
        WHEN trip_miles < 10 THEN '5-10 mi'\n
        ELSE '10+ mi'\n
    END AS distance_bucket,\n
    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n
    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n
    COUNT(*) AS rows\n
FROM predictions_dt\n
GROUP BY 1\n
ORDER BY distance_bucket\n
""").show(truncate=False)

In [ ]:
tree_stage = dt_model.stages[-1]\n
importance_pdf = pd.DataFrame({\n
    'feature': feature_cols,\n
    'importance': list(tree_stage.featureImportances)\n
}).sort_values('importance', ascending=False)\n
\n
plt.figure(figsize=(10, 6))\n
sns.barplot(data=importance_pdf.head(15), x='importance', y='feature')\n
plt.title('Top 15 importancias - Decision Tree')\n
plt.tight_layout()\n
plt.savefig(IMPORTANCE_PLOT, dpi=150, bbox_inches='tight')\n
plt.show()\n
\n
importance_pdf.head(15)

In [ ]:
results_df = append_metric(
    model='decision_tree',
    rmse=metrics['rmse'],
    mae=metrics['mae'],
    r2=metrics['r2'],
    train_seconds=train_seconds,
    notes='DecisionTreeRegressor maxDepth=8 maxBins=256',
)
results_df.tail(10)

In [ ]:
spark.stop()